In [1]:
import numpy as np
import pandas as pd 
import sqlite3
import requests

from tqdm import tqdm
from xml.etree import ElementTree
from IPython.display import HTML

from pprint import pprint

pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

In [2]:
#!pip install openpyxl
#!pip install tqdm
#!pip install tabulate

In [3]:
# !proxyon

In [32]:
# Run in /nfs/zefys (takes roughly 24 hours!)
# !find ./ -wholename "*/presentation/*.jpg" -o -wholename "*/presentation/*.jpeg" -o -wholename "*/presentation/*.png" > ~/SPUNK/workbench/zefys_image_files.txt

In [5]:
!ls

README.md  statistics.html  statistics.md  workbench  zefys-statistics.ipynb


In [6]:
!pwd

/nfs/git-annex/kai.labusch/SPUNK/SPUNK


In [7]:
df_all = pd.read_csv('workbench/zefys_image_files.txt')

In [8]:
df = df_all.loc[~df_all.fullpath.str.startswith('./scandata') & ~df_all.fullpath.str.startswith('./_tosort')].copy() # remove scandata files since they do not follow the schema

In [9]:
df[['zdb', 'year', 'month', 'day', 'issue']]= df.fullpath.str.extract('./[publish/]*([^/]+)/([^/]+)/([^/]+)/([^/]+)/([^/]+)/.*')

In [10]:
df = df.dropna()

In [11]:
df[['zdb']] = df.zdb.str.extract('([^_]+).*')

In [12]:
df[['page','type']] = df.fullpath.str.extract('.*?([0-9]+).(png|jpg|jpeg|tif)$')

In [13]:
df.page = df.page.astype(int)

df = df.loc[df.page > 0]

df.issue = df.issue.astype(int)

In [14]:
df.shape

(6462694, 8)

In [15]:
df_meta=[]
for _, (zdb,count) in tqdm(pd.DataFrame(df.zdb.value_counts().reset_index()).iterrows()):
    zdb_id = zdb[0:-1] + "-" +  zdb[-1:]

    url = "https://services.dnb.de/sru/zdb?version=1.1&operation=searchRetrieve&query=zdbid={}&recordSchema=oai_dc".format(zdb_id)

    response = requests.get(url, stream=True)

    response.raw.decode_content = True

    events = ElementTree.iterparse(response.raw)

    meta = {"zdb": zdb, "title": "", "creator": "", "publisher": "", "date": "", "language": ""}

    for event, elem in events:
        # print(elem.tag, elem.text)

        if elem.tag=="{http://purl.org/dc/elements/1.1/}title":
            meta["title"]= elem.text
            continue

        if elem.tag=="{http://purl.org/dc/elements/1.1/}creator":
            meta["creator"]= elem.text
            continue

        if elem.tag=="{http://purl.org/dc/elements/1.1/}publisher":
            meta["publisher"]= elem.text
            continue

        if elem.tag=="{http://purl.org/dc/elements/1.1/}date":
            meta["date"]= elem.text
            continue

        if elem.tag=="{http://purl.org/dc/elements/1.1/}language":
            meta["language"]= elem.text
            continue
            
    df_meta.append(meta)
    
    # break

df_meta = pd.DataFrame(df_meta)


216it [00:23,  9.16it/s]


In [16]:
df_meta.title = df_meta.title.str.extract('([^:]*).*?')

In [17]:
df_tmp = df_meta.merge(df, on='zdb')

In [18]:
df_tmp.shape

(6462694, 13)

In [19]:
df_list = pd.DataFrame(df_tmp.zdb.value_counts()).reset_index()
df_list["cumsum"] = df_list["count"].cumsum()
df_list=df_list[["count", "cumsum", "zdb"]].merge(df_meta, on="zdb")

In [20]:
df_list

,count,cumsum,zdb,title,creator,publisher,date,language
0,706544,706544,24353991,Königlich privilegirte Berlinische Zeitung von Staats- und gelehrten Sachen,"Fontane, Theodor [Mitwirkender]",Berlin : Voss. Erben,1827-1911,ger
1,641669,1348213,2436020X,Berliner Börsen-Zeitung,,Berlin : Metzold,1857-1939,ger
2,612275,1960488,27646518,Berliner Tageblatt und Handels-Zeitung,"Tergit, Gabriele [Mitwirkender]",Berlin : Mosse,1872-1932,ger
3,310273,2270761,24340492,National-Zeitung,,Berlin : Die Expedition der National-Zeitung,1848-1910,ger
4,291734,2562495,30744556,Tägliche Rundschau,,Berlin : Zeitungsverlag Schmidt-Dumont & Co.,1881-1922,ger
5,281899,2844394,2719372X,Berliner Morgenpost,,Berlin : Ullstein,1898-1933,ger
6,254113,3098507,27913090,Berlinische Nachrichten von Staats- und gelehrten Sachen,,Berlin : Haude & Spener,1740-1872,ger
7,203312,3301819,30743977,Deutsche Tageszeitung,,Berlin : Dt. Tageszeitung AG,1894-1920,ger
8,195156,3496975,3074409X,Der Tag,,Berlin : Scherl,1901-1921,ger
9,181813,3678788,26120215,Berliner Zeitung Online-Archiv,,Berlin : Berliner Verl.,1945-1990,ger


In [21]:
print(df_list.to_markdown())

|     |   count |   cumsum | zdb          | title                                                                                                                                                                           | creator                                          | publisher                                                                     | date      | language   |
|----:|--------:|---------:|:-------------|:--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|:-------------------------------------------------|:------------------------------------------------------------------------------|:----------|:-----------|
|   0 |  706544 |   706544 | 24353991     | Königlich privilegirte Berlinische Zeitung von Staats- und gelehrten Sachen                                                                                                     | Fontane, Theodor [Mitw

In [31]:
with open("statistics.md", "w") as f:
    f.write(df_list.to_markdown())

In [22]:
df_list_html = df_list.to_html(index=False, border=2)

html_template = f"""
<html>
<head>
    <style>
        table {{
            width: 60%;
            border-collapse: collapse;
            margin: 20px auto;
        }}
        th, td {{
            border: 1px solid black;
            padding: 8px;
            text-align: center;
        }}
        th {{
            background-color: #4CAF50;
            color: white;
        }}
        tr:nth-child(even) {{
            background-color: #f2f2f2;
        }}
    </style>
</head>
<body>
    {df_list_html}
</body>
</html>
"""


with open("statistics.html", "w") as f:
    f.write(html_template)

In [23]:
df_rest = df_meta.merge(df.zdb.value_counts(), left_on='zdb', right_index=True, how='right')
df_rest = df_rest.loc[df_rest.title.isnull()][['zdb','count']].reset_index(drop=True)
df_rest

,zdb,count


In [24]:
df_rest['count'].sum()

np.int64(0)

In [25]:
pd.DataFrame(df_tmp.type.value_counts())

,count
type,
jpg,5953395
png,509299


In [26]:
df_tmp['url'] = "https://content.staatsbibliothek-berlin.de/zefys/SNP" + df_tmp.zdb + "-" + df_tmp.year + df_tmp.month + df_tmp.day + "-" + (df_tmp.issue - 1).astype(str) + "-" + df_tmp.page.astype(str) + "-0-0/full/full/0/default.jpg"

In [29]:
df_tmp.reset_index(drop=True).to_csv('workbench/zefys-files.tsv', sep='\t')

In [28]:
df_tmp.head(2887947).tail()

,zdb,title,creator,publisher,date,language,fullpath,year,month,day,issue,page,type,url
2887942,27913090,Berlinische Nachrichten von Staats- und gelehrten Sachen,,Berlin : Haude & Spener,1740-1872,ger,./publish/27913090_01/1832/02/02/01/presentation/27913090_1832-02-02_000_28_H_1_007.jpg,1832,02,02,1,7,jpg,https://content.staatsbibliothek-berlin.de/zefys/SNP27913090-18320202-0-7-0-0/full/full/0/default.jpg
2887943,27913090,Berlinische Nachrichten von Staats- und gelehrten Sachen,,Berlin : Haude & Spener,1740-1872,ger,./publish/27913090_01/1832/02/02/01/presentation/27913090_1832-02-02_000_28_H_1_012.jpg,1832,02,02,1,12,jpg,https://content.staatsbibliothek-berlin.de/zefys/SNP27913090-18320202-0-12-0-0/full/full/0/default.jpg
2887944,27913090,Berlinische Nachrichten von Staats- und gelehrten Sachen,,Berlin : Haude & Spener,1740-1872,ger,./publish/27913090_01/1832/02/02/01/presentation/27913090_1832-02-02_000_28_H_1_016.jpg,1832,02,02,1,16,jpg,https://content.staatsbibliothek-berlin.de/zefys/SNP27913090-18320202-0-16-0-0/full/full/0/default.jpg
2887945,27913090,Berlinische Nachrichten von Staats- und gelehrten Sachen,,Berlin : Haude & Spener,1740-1872,ger,./publish/27913090_01/1832/02/02/01/presentation/27913090_1832-02-02_000_28_H_1_004.jpg,1832,02,02,1,4,jpg,https://content.staatsbibliothek-berlin.de/zefys/SNP27913090-18320202-0-4-0-0/full/full/0/default.jpg
2887946,27913090,Berlinische Nachrichten von Staats- und gelehrten Sachen,,Berlin : Haude & Spener,1740-1872,ger,./publish/27913090_01/1832/02/02/01/presentation/27913090_1832-02-02_000_28_H_1_005.jpg,1832,02,02,1,5,jpg,https://content.staatsbibliothek-berlin.de/zefys/SNP27913090-18320202-0-5-0-0/full/full/0/default.jpg
